# 10 - Learning-Based Target Selection
Train and evaluate lightweight classifiers (RF, MLP, GBDT) to
learn which boundary loop is the removal-induced target.

This addresses the editor's suggestion:
> *extend the method beyond geometric baselines with learning-aware
> boundary selection or lightweight geometric learning*

**Prerequisite**: Run notebook 01 first to build the dataset.

In [1]:
import sys, os, gc
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import load_config, ensure_dirs
from src.data.dataset_index import DatasetIndex
from src.data.sample_loader import SampleLoader
from src.geometry.boundary import extract_boundary_loops
from src.target_selection.learning import (
    collect_training_data, LoopClassifierRF, LoopClassifierMLP,
    LoopClassifierGBDT, select_loops_by_classifier,
)
from src.target_selection.selectors import select_target_loops_by_bbox
from src.repair.planar_patch import planar_triangulation_repair
from src.evaluation.evaluator import Evaluator
from tqdm import tqdm

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
cfg = load_config(os.path.join(PROJECT_ROOT, 'configs', 'chair_leg.yaml'))
ensure_dirs(cfg)
index = DatasetIndex(cfg['paths']['raw_data_dir'])
margin = cfg['repair']['margin']
prox_thresh = cfg['repair']['proximity_threshold']
print(f'Dataset: {len(index)} samples')

Dataset: 100 samples


In [3]:
# ============================================================
# STEP 1: Collect training data (features + labels for all loops)
# ============================================================
print('Collecting training data from all samples...')
features_list, labels = collect_training_data(
    sample_ids=index.sample_ids,
    data_dir=cfg['paths']['raw_data_dir'],
    margin=margin,
    threshold=prox_thresh,
)

n_pos = sum(labels)
n_neg = len(labels) - n_pos
print(f'Total loops: {len(labels)}')
print(f'  Target (positive): {n_pos}')
print(f'  Non-target (negative): {n_neg}')
print(f'  Imbalance ratio: 1:{n_neg / max(n_pos, 1):.1f}')

Total loops: 7833
  Target (positive): 3675
  Non-target (negative): 4158
  Imbalance ratio: 1:1.1


In [4]:
# ============================================================
# STEP 2: Cross-validate all three classifiers
# ============================================================
classifiers = {
    'RandomForest': LoopClassifierRF(n_estimators=100),
    'MLP': LoopClassifierMLP(hidden_sizes=(64, 32)),
    'GBDT': LoopClassifierGBDT(n_estimators=100),
}

cv_results = {}
for name, clf in classifiers.items():
    print(f'\nCross-validating {name}...')
    scores = clf.cross_validate(features_list, labels, cv=5)
    cv_results[name] = scores
    print(f'  F1 score: {scores["f1_mean"]:.4f} ± {scores["f1_std"]:.4f}')

print('\n--- Summary ---')
for name, scores in cv_results.items():
    print(f'  {name:<15} F1 = {scores["f1_mean"]:.4f} ± {scores["f1_std"]:.4f}')


Cross-validating RandomForest...
  F1 score: 0.9462 ± 0.0462

Cross-validating MLP...
  F1 score: 0.9537 ± 0.0371

Cross-validating GBDT...
  F1 score: 0.9471 ± 0.0461

--- Summary ---
  RandomForest    F1 = 0.9462 ± 0.0462
  MLP             F1 = 0.9537 ± 0.0371
  GBDT            F1 = 0.9471 ± 0.0461


In [5]:
# ============================================================
# STEP 3: Train best classifier on full data
# ============================================================
# Pick the best by F1
best_name = max(cv_results, key=lambda k: cv_results[k]['f1_mean'])
print(f'Best classifier: {best_name}')

best_clf = classifiers[best_name]
best_clf.fit(features_list, labels)

# Feature importance (RF / GBDT)
if hasattr(best_clf, 'feature_importance'):
    imp = best_clf.feature_importance()
    if imp:
        print('\nFeature importance:')
        for k, v in sorted(imp.items(), key=lambda x: -x[1]):
            print(f'  {k:<25} {v:.4f}')

# Save model
model_path = os.path.join(cfg['paths']['output_dir'], 'models', f'loop_clf_{best_name}.pkl')
best_clf.save(model_path)
print(f'\nModel saved to {model_path}')

Best classifier: MLP

Model saved to D:\MyJupyter\Works\3DPART_v2\outputs\models\loop_clf_MLP.pkl


In [6]:
# ============================================================
# STEP 4: End-to-end evaluation with learned selection
# Compare: Geometric RPA vs Learned selection vs LH-only
# ============================================================
loader = SampleLoader(cfg['paths']['raw_data_dir'])
evaluator = Evaluator(margin=margin, proximity_threshold=prox_thresh)

results_rpa = []
results_learned = []
results_lh = []

for sid in tqdm(index.sample_ids, desc='Evaluating'):
    try:
        sample = loader.load(sid)
        damaged = sample['damaged_mesh']
        removed = sample['removed_part_mesh']
        loops = extract_boundary_loops(damaged)
        if not loops:
            continue

        # Ground truth target (RPA)
        targets_rpa = select_target_loops_by_bbox(
            damaged, loops, removed, margin, prox_thresh)

        # Learned target selection
        targets_learned = select_loops_by_classifier(
            damaged, loops, removed, best_clf, prob_threshold=0.5)

        # Largest-hole-only
        from src.target_selection.selectors import select_largest_loop
        targets_lh = select_largest_loop(damaged, loops)

        # Repair and evaluate with planar triangulation
        for targets, result_list in [
            (targets_rpa, results_rpa),
            (targets_learned, results_learned),
            (targets_lh, results_lh),
        ]:
            res = planar_triangulation_repair(damaged, targets)
            metrics = evaluator.evaluate(
                damaged, res['repaired_mesh'], removed,
                res, targets_rpa)  # always eval against RPA ground truth
            metrics['sample_id'] = sid
            result_list.append(metrics)
            del res

    except Exception as e:
        print(f'  Error on {sid}: {e}')

    if len(results_rpa) % 20 == 0:
        gc.collect()

df_rpa = pd.DataFrame(results_rpa)
df_learned = pd.DataFrame(results_learned)
df_lh = pd.DataFrame(results_lh)

Evaluating: 100%|████████████████████████████████████████████████████████████████████| 100/100 [07:19<00:00,  4.39s/it]


In [7]:
# ============================================================
# STEP 5: Compare results
# ============================================================
metrics_to_compare = [
    ('target_loop_length_after', 'Residual ↓', True),
    ('improvement', 'Improvement ↑', False),
    ('locality_ratio', 'Locality ↑', False),
    ('avg_new_face_quality', 'Quality ↑', False),
]

print(f'{"":<30} {"Geom. RPA":>12} {"Learned":>12} {"LH-Only":>12}')
print('=' * 70)
for col, name, lower_better in metrics_to_compare:
    rv = df_rpa[col].mean() if col in df_rpa.columns else float('nan')
    lv = df_learned[col].mean() if col in df_learned.columns else float('nan')
    hv = df_lh[col].mean() if col in df_lh.columns else float('nan')
    print(f'{name:<30} {rv:>12.4f} {lv:>12.4f} {hv:>12.4f}')

                                  Geom. RPA      Learned      LH-Only
Residual ↓                           1.9384       1.9480      38.2569
Improvement ↑                       38.0133      38.0036       1.6948
Locality ↑                           0.6710       0.6708       0.5717
Quality ↑                            0.4341       0.4343       0.4122


In [8]:
# ============================================================
# STEP 6: Visualize comparison
# ============================================================
from src.visualization.plot_utils import set_paper_style
set_paper_style()

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
strategy_names = ['Geom. RPA', f'Learned ({best_name})', 'LH-Only']
colors = ['#70AD47', '#A855F7', '#FFC000']

for ax, (col, title, _) in zip(axes, metrics_to_compare):
    data = [
        df_rpa[col].dropna().values if col in df_rpa.columns else [],
        df_learned[col].dropna().values if col in df_learned.columns else [],
        df_lh[col].dropna().values if col in df_lh.columns else [],
    ]
    bp = ax.boxplot(data, labels=strategy_names, patch_artist=True, widths=0.5)
    for patch, c in zip(bp['boxes'], colors):
        patch.set_facecolor(c)
    ax.set_title(title, fontsize=10)

plt.suptitle('Target Selection Strategy Comparison', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(cfg['paths']['figures_dir'], 'fig_learning_comparison.pdf'),
            dpi=300, bbox_inches='tight')
plt.show()
print('\nDone! Learning-based selection experiment complete.')

C:\Users\Administrator\AppData\Local\Temp\ipykernel_18636\1572745375.py:17: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data, labels=strategy_names, patch_artist=True, widths=0.5)
C:\Users\Administrator\AppData\Local\Temp\ipykernel_18636\1572745375.py:17: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data, labels=strategy_names, patch_artist=True, widths=0.5)
C:\Users\Administrator\AppData\Local\Temp\ipykernel_18636\1572745375.py:17: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data, labels=strategy_names, patch_artist=True, widths=0.5)
C:\Users\Administrator\AppData\


Done! Learning-based selection experiment complete.


C:\Users\Administrator\AppData\Local\Temp\ipykernel_18636\1572745375.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
